In [2]:
import pandas as pd
import numpy as np
import os 
import datetime 
import pandas as pd
import anndata as ad

In [4]:
# read in the metadata file 
metadata = pd.read_csv("/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/zenodo/cell_metadata_cols.tsv", sep="\t")

# read in file containing bam file IDs to match with metadata
ftp_data = pd.read_csv("/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/RAW/filereport_read_run_PRJEB14362_tsv.txt", sep="\t")

# Extract the cell name from the `submitted_ftp` column
ftp_data["cell_name"] = ftp_data["submitted_ftp"].str.split(";").str[0].str.split("/").str[-1].str.split(".").str[0]
ftp_data = ftp_data[["run_accession", "cell_name", "library_name", "sample_title"]].drop_duplicates()

# metadata = metadata.merge(ftp_data, on = "cell_name")
# make new column in metadata called cell_id using metadata["run_accession"] 
# metadata["cell_id"] = metadata["run_accession"]


In [6]:
metadata = metadata[["cell_name", "donor", "day", "pseudo", "experiment", "plate_id", "plate_well_id", "donor_short_id", "donor_long_id"]].drop_duplicates()
print(metadata.head())

# print the excel file containg cell line information 
cell_lines = "/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/RAW/41467_2020_14457_MOESM4_ESM.txt"
cell_lines = pd.read_csv(cell_lines, sep="\t")

               cell_name donor   day    pseudo experiment  plate_id  \
21843_1#10    21843_1#10  joxm  day1  0.292682    expt_09       126   
21843_1#100  21843_1#100  fafq  day1  0.484716    expt_09       126   
21843_1#101  21843_1#101  fafq  day1  0.403809    expt_09       126   
21843_1#102  21843_1#102  wuye  day1  0.260772    expt_09       126   
21843_1#103  21843_1#103  joxm  day1  0.355366    expt_09       126   

            plate_well_id donor_short_id     donor_long_id  
21843_1#10       0126_A10         joxm_1  HPSI0114i-joxm_1  
21843_1#100      0126_E04         fafq_1  HPSI0314i-fafq_1  
21843_1#101      0126_E05         fafq_1  HPSI0314i-fafq_1  
21843_1#102      0126_E06         wuye_2  HPSI1013i-wuye_2  
21843_1#103      0126_E07         joxm_1  HPSI0114i-joxm_1  


In [7]:
metadata.shape, cell_lines.shape

((36044, 9), (126, 6))

In [8]:
# rename cell_lines column cell_line_id to donor_long_id 
cell_lines = cell_lines.rename(columns = {"cell_line_id": "donor_long_id"})

In [11]:
metadata_df = cell_lines.merge(metadata, on = "donor_long_id")
metadata_df.head()

,donor_long_id,donor_id_short,sex,donor_disease_status,n_cells,experiments,cell_name,donor,day,pseudo,experiment,plate_id,plate_well_id,donor_short_id
0,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#106,fejf,day0,0.109739,expt_35,2806,2806_E10,fejf_2
1,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#117,fejf,day0,0.038109,expt_35,2806,2806_E21,fejf_2
2,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#135,fejf,day0,0.067188,expt_35,2806,2806_F15,fejf_2
3,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#147,fejf,day0,0.119335,expt_35,2806,2806_G03,fejf_2
4,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#158,fejf,day0,0.097844,expt_35,2806,2806_G14,fejf_2


In [10]:
WD = "/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/zenodo"

In [12]:
# Load the normalized counts
log_counts_path = f"{WD}/log_normalised_counts.csv.zip"
log_counts = pd.read_csv(log_counts_path, index_col=0)

In [13]:
# Load the raw counts (optional: only if you want raw data)
raw_counts_path = f"{WD}/raw_counts.csv.zip"
raw_counts = pd.read_csv(raw_counts_path, index_col=0)

In [14]:
metadata_df

,donor_long_id,donor_id_short,sex,donor_disease_status,n_cells,experiments,cell_name,donor,day,pseudo,experiment,plate_id,plate_well_id,donor_short_id
0,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#106,fejf,day0,0.109739,expt_35,2806,2806_E10,fejf_2
1,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#117,fejf,day0,0.038109,expt_35,2806,2806_E21,fejf_2
2,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#135,fejf,day0,0.067188,expt_35,2806,2806_F15,fejf_2
3,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#147,fejf,day0,0.119335,expt_35,2806,2806_G03,fejf_2
4,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#158,fejf,day0,0.097844,expt_35,2806,2806_G14,fejf_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36039,HPSI0614i-qunz_3,qunz,female,normal,293,expt_43,25535_6#60,qunz,day1,0.352654,expt_43,3351,3351_C12,qunz_3
36040,HPSI0614i-qunz_3,qunz,female,normal,293,expt_43,25535_6#85,qunz,day1,0.306219,expt_43,3351,3351_D13,qunz_3
36041,HPSI0614i-qunz_3,qunz,female,normal,293,expt_43,25535_6#87,qunz,day1,0.323420,expt_43,3351,3351_D15,qunz_3
36042,HPSI0614i-qunz_3,qunz,female,normal,293,expt_43,25535_6#95,qunz,day1,0.393949,expt_43,3351,3351_D23,qunz_3


In [15]:
log_counts.shape

(11231, 36044)

In [16]:
# match metadata_df "cell_name" with columns of log_counts and raw_counts
log_counts_sub = log_counts[metadata_df["cell_name"]]
raw_counts_sub = raw_counts[metadata_df["cell_name"]]

# ensure metadata_df["cell_name"] order matches the columns of log_counts and raw_counts
assert (metadata_df["cell_name"] == log_counts_sub.columns).all()
assert (metadata_df["cell_name"] == raw_counts_sub.columns).all()

In [17]:
# print their shapes and ensure they match
log_counts_sub.shape, raw_counts_sub.shape

((11231, 36044), (11231, 36044))

In [18]:
# ensure Index of obs must match index of X. 
metadata_df.index = metadata_df["cell_name"]

In [19]:
# Create the AnnData object
adata = ad.AnnData(X=log_counts_sub.T, obs=metadata_df, var=pd.DataFrame(index=log_counts_sub.index))

# Add a column to adata.var to indicate the gene names, split index into ensg and gene_name by "_"
adata.var["ensg"] = adata.var.index.str.split("_").str[0]
adata.var["gene_name"] = adata.var.index.str.split("_").str[1]

# Optional: Add raw counts to the AnnData object
adata.raw = adata.copy()
adata.layers["raw_counts"] = raw_counts_sub.T

In [21]:
adata.obs

,donor_long_id,donor_id_short,sex,donor_disease_status,n_cells,experiments,cell_name,donor,day,pseudo,experiment,plate_id,plate_well_id,donor_short_id
cell_name,,,,,,,,,,,,,,
24229_3#106,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#106,fejf,day0,0.109739,expt_35,2806,2806_E10,fejf_2
24229_3#117,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#117,fejf,day0,0.038109,expt_35,2806,2806_E21,fejf_2
24229_3#135,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#135,fejf,day0,0.067188,expt_35,2806,2806_F15,fejf_2
24229_3#147,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#147,fejf,day0,0.119335,expt_35,2806,2806_G03,fejf_2
24229_3#158,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35,24229_3#158,fejf,day0,0.097844,expt_35,2806,2806_G14,fejf_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25535_6#60,HPSI0614i-qunz_3,qunz,female,normal,293,expt_43,25535_6#60,qunz,day1,0.352654,expt_43,3351,3351_C12,qunz_3
25535_6#85,HPSI0614i-qunz_3,qunz,female,normal,293,expt_43,25535_6#85,qunz,day1,0.306219,expt_43,3351,3351_D13,qunz_3
25535_6#87,HPSI0614i-qunz_3,qunz,female,normal,293,expt_43,25535_6#87,qunz,day1,0.323420,expt_43,3351,3351_D15,qunz_3


In [23]:
# Assign cells developmental stages based on the following rules:
# - Eveyrything below 0.15 is iPSC that is day 1 or 2 is iPSC
# - Cells were assigned to the mesendo stage if they were collected at day1 or day2, and had pseudotime values between 0.15 and 0.5, corresponding to a pseudotime window around the peak expression of Brachyury (T), a marker of mesendoderm (Supplementary Fig. 8A). 
# - Cells were assigned to the defendo stage if they were collected at day2 or day3, and had pseudotime values higher than 0.7, corresponding to a pseudotime window with maximal expression of GATA6, a marker of definitive endoderm (Supplementary Fig. 8B). 
# - Cells with intermediate pseudotime (between 0.5 and 0.7) mostly came from day2, and were not assigned to any stage for the purposes of the initial stage QTL mapping (results shown in Fig. 2, Supplementary Table 1). Overall, we assign 28,971 (80%) cells to any of the stages (iPSC, mesendo, defendo).

adata.obs["dev_stage"] = "" 
adata.obs.loc[(adata.obs["day"] == 1) | (adata.obs["day"] == 2) | (adata.obs["pseudo"] < 0.15), "dev_stage"] = "iPSC"
adata.obs.loc[(adata.obs["day"] == 1) | (adata.obs["day"] == 2) | ((adata.obs["pseudo"] >= 0.15) & (adata.obs["pseudo"] <= 0.5)), "dev_stage"] = "mesendo"
adata.obs.loc[(adata.obs["day"] == 2) | (adata.obs["day"] == 3) | (adata.obs["pseudo"] > 0.7), "dev_stage"] = "defendo"

# remaining cells just write "intermediate"
adata.obs.loc[adata.obs["dev_stage"] == "", "dev_stage"] = "int_unassigned"
adata.obs.dev_stage.value_counts()

dev_stage
mesendo           13895
defendo            9908
iPSC               6455
int_unassigned     5786
Name: count, dtype: int64

In [24]:
13895+9908+6455

30258

In [25]:
# Save the AnnData object for future use in WD with today's date 
today = datetime.datetime.today().strftime('%Y-%m-%d')
adata.write(f"{WD}/gene_expression_{today}.h5ad")

In [26]:
adata.obs.donor_disease_status.value_counts()

donor_disease_status
normal               34337
neonatal_diabetes     1707
Name: count, dtype: int64

In [27]:
adata.obs.donor.value_counts()

donor
joxm    1415
guss    1093
poih    1077
nudd     919
sojd     880
        ... 
zihe      26
suop      17
tavh      12
lexy       7
hiaf       4
Name: count, Length: 125, dtype: int64

In [28]:
cell_lines

,donor_long_id,donor_id_short,sex,donor_disease_status,n_cells,experiments
0,HPSI0513i-fejf_2,fejf,male,normal,155,expt_35
1,HPSI0613i-hegp_3,hegp,female,normal,76,expt_31
2,HPSI0713i-nocf_2,nocf,male,normal,615,expt_32;expt_35
3,HPSI0613i-ueah_1,ueah,male,normal,134,expt_31
4,HPSI0413i-nudd_1,nudd,male,normal,919,expt_33;expt_34
...,...,...,...,...,...,...
121,HPSI0914i-suop_5,suop,male,normal,17,expt_30
122,HPSI0514i-uenn_3,uenn,male,normal,95,expt_37
123,HPSI0115i-aoxv_3,aoxv,female,normal,293,expt_43
124,HPSI0514i-tert_1,tert,male,normal,355,expt_45


In [29]:
metadata

,cell_name,donor,day,pseudo,experiment,plate_id,plate_well_id,donor_short_id,donor_long_id
21843_1#10,21843_1#10,joxm,day1,0.292682,expt_09,126,0126_A10,joxm_1,HPSI0114i-joxm_1
21843_1#100,21843_1#100,fafq,day1,0.484716,expt_09,126,0126_E04,fafq_1,HPSI0314i-fafq_1
21843_1#101,21843_1#101,fafq,day1,0.403809,expt_09,126,0126_E05,fafq_1,HPSI0314i-fafq_1
21843_1#102,21843_1#102,wuye,day1,0.260772,expt_09,126,0126_E06,wuye_2,HPSI1013i-wuye_2
21843_1#103,21843_1#103,joxm,day1,0.355366,expt_09,126,0126_E07,joxm_1,HPSI0114i-joxm_1
...,...,...,...,...,...,...,...,...,...
24539_8#93,24539_8#93,nocf,day1,0.199677,expt_35,2808,2808_D21,nocf_2,HPSI0713i-nocf_2
24539_8#94,24539_8#94,zagm,day1,0.251769,expt_35,2808,2808_D22,zagm_1,HPSI1013i-zagm_1
24539_8#95,24539_8#95,wigw,day1,0.226935,expt_35,2808,2808_D23,wigw_2,HPSI0314i-wigw_2
24539_8#97,24539_8#97,wahn,day1,0.389748,expt_35,2808,2808_E01,wahn_1,HPSI1113i-wahn_1


In [30]:
metadata_full = pd.read_csv("/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/zenodo/cell_metadata_cols.tsv", sep="\t")
metadata_full

,assigned,auxDir,cell_filter,cell_name,compatible_fragment_ratio,day,donor,expected_format,experiment,frag_dist_length,...,donor_short_id,donor_long_id,pseudo,PC1_top100hvgs,PC1_top200hvgs,PC1_top500hvgs,PC1_top1000hvgs,PC1_top2000hvgs,princ_curve,princ_curve_scaled01
21843_1#10,1,aux_info,True,21843_1#10,0.999497,day1,joxm,IU,expt_09,1001,...,joxm_1,HPSI0114i-joxm_1,0.292682,-13.353833,-12.969161,-11.769526,-12.153335,-12.871002,39.972588,0.352570
21843_1#100,1,aux_info,True,21843_1#100,0.999456,day1,fafq,IU,expt_09,1001,...,fafq_1,HPSI0314i-fafq_1,0.484716,2.399795,4.633188,5.131531,7.883242,9.916888,56.433387,0.497759
21843_1#101,1,aux_info,True,21843_1#101,0.999549,day1,fafq,IU,expt_09,1001,...,fafq_1,HPSI0314i-fafq_1,0.403809,-0.612621,0.006692,-0.643021,0.185298,-0.425978,52.966142,0.467177
21843_1#102,1,aux_info,True,21843_1#102,0.999422,day1,wuye,IU,expt_09,1001,...,wuye_2,HPSI1013i-wuye_2,0.260772,-11.946009,-12.691233,-14.508021,-15.328236,-18.071351,38.419984,0.338876
21843_1#103,1,aux_info,True,21843_1#103,0.999446,day1,joxm,IU,expt_09,1001,...,joxm_1,HPSI0114i-joxm_1,0.355366,-5.735735,-5.389405,-5.802985,-6.625860,-9.430127,48.488060,0.427679
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24539_8#93,1,aux_info,True,24539_8#93,0.999667,day1,nocf,IU,expt_35,1001,...,nocf_2,HPSI0713i-nocf_2,0.199677,-16.077636,-18.294428,-22.442847,-26.444635,-31.479645,33.905511,0.299057
24539_8#94,1,aux_info,True,24539_8#94,0.999848,day1,zagm,IU,expt_35,1001,...,zagm_1,HPSI1013i-zagm_1,0.251769,-12.789361,-14.500027,-18.627453,-23.003981,-27.443642,36.894253,0.325418
24539_8#95,1,aux_info,True,24539_8#95,0.999633,day1,wigw,IU,expt_35,1001,...,wigw_2,HPSI0314i-wigw_2,0.226935,-12.115839,-13.863510,-17.766493,-24.104888,-30.390688,33.272989,0.293478
24539_8#97,1,aux_info,True,24539_8#97,0.999816,day1,wahn,IU,expt_35,1001,...,wahn_1,HPSI1113i-wahn_1,0.389748,-11.146346,-11.620859,-13.033028,-15.120121,-17.781083,40.862154,0.360416


In [31]:
# metadata_full.shape --> # 36044 in gene expression matrix as well but their cell names don't match 

In [33]:
row_dict = metadata_full.iloc[1].to_dict()
for key, value in row_dict.items():
    print(f"{key}: {value}")


assigned: 1
auxDir: aux_info
cell_filter: True
cell_name: 21843_1#100
compatible_fragment_ratio: 0.99945606112355
day: day1
donor: fafq
expected_format: IU
experiment: expt_09
frag_dist_length: 1001
gc_bias_correct: True
is_cell_control: False
is_cell_control_bulk: False
is_cell_control_control: False
library_types: IU
libType: IU
log10_total_counts: 5.34744674008841
log10_total_counts_endogenous: 5.28937682161842
log10_total_counts_ERCC: 3.09271056492959
log10_total_counts_feature_control: 4.4449177529445
log10_total_counts_MT: 4.42519115768671
log10_total_features: 3.96553094362286
log10_total_features_endogenous: 3.96411814315148
log10_total_features_ERCC: 1.25527250510331
log10_total_features_feature_control: 1.49136169383427
log10_total_features_MT: 1.14612803567824
mapping_type: mapping
mates1: data_raw/scrnaseq/run_21843/fastq/21843_1#100_1_val_1.fq.gz
mates2: data_raw/scrnaseq/run_21843/fastq/21843_1#100_2_val_2.fq.gz
n_alt_reads: 2515
n_total_reads: 9200
num_assigned_fragments